# CIFAR-10 — Design a CNN

**Your task:** develop a convolutional neural network architecture for ten-class image classification. Edit the `model.py` cell, train your model, compare validation accuracy, then export your model for the arena's **test accuracy leaderboard**.

Repository: [jamie-duell/interactive_coding-main](https://github.com/jamie-duell/interactive_coding-main).

Choose **Runtime → Change runtime type → T4 GPU** if available. Save a copy of this notebook to your Drive before working.

Train from scratch using only CIFAR-10 training images. Do not use pretrained weights, official test labels, or label lookup tables. Keep the official test split for server evaluation; use validation accuracy to choose architectures. The official test labels are public, so the exercise relies on classroom trust.

[Dataset: Krizhevsky, Learning Multiple Layers of Features from Tiny Images (2009)](https://www.cs.toronto.edu/~kriz/cifar.html).

In [ ]:
import copy
import importlib
import random
from pathlib import Path
import zipfile

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)
CLASSES = ("airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")

## Load data and reserve validation images

Use 45,000 training images and 5,000 validation images from the official **training** split. Both use RGB tensors with shape `(3, 32, 32)`, scaled to `[0, 1]` and normalised using mean and standard deviation `(0.5, 0.5, 0.5)`. Random crops and flips apply only during training.

The server uses exactly the deterministic validation preprocessing. If you change it, also update the preprocessing in the export cell.

In [ ]:
normalise = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), normalise,
])
validation_transform = transforms.Compose([transforms.ToTensor(), normalise])
train_data = datasets.CIFAR10("data", train=True, download=True, transform=train_transform)
validation_data = datasets.CIFAR10("data", train=True, download=False, transform=validation_transform)
indices = torch.randperm(len(train_data), generator=torch.Generator().manual_seed(SEED)).tolist()
train_set = Subset(train_data, indices[5000:])
validation_set = Subset(validation_data, indices[:5000])
BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
validation_loader = DataLoader(validation_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"{len(train_set):,} training images; {len(validation_set):,} validation images")

## Your architecture — edit this cell

The starter is a small working CNN. Develop your own architecture by changing the number of blocks, convolutional filters, kernel sizes, pooling, batch normalisation, dropout, and classifier. The input is `(batch, 3, 32, 32)`; the output must be `(batch, 10)` **logits**, without softmax during training.

Keep the class name `CNN` and allow `CNN()` without arguments. Put custom layers and helper functions in this file too. Use PyTorch and standard Python here: the inference runner does not include torchvision. After each architecture edit, rerun model creation and training before export.

**Explain your design:** what changes will you make, and why should they help?

In [ ]:
%%writefile model.py
import torch
from torch import nn


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Develop your own convolutional blocks and classifier.
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

In [ ]:
# Reload the file after every architecture edit, then initialise fresh weights.
import model as architecture
importlib.reload(architecture)
model = architecture.CNN().to(device)
trained_architecture = Path("model.py").read_text()
assert model(torch.zeros(2, 3, 32, 32, device=device)).shape == (2, 10)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Train and validate

Start with 5 epochs to check your pipeline. Increase epochs when comparing promising architectures. You can also change optimiser, learning rate, weight decay, augmentation and schedules. The best model is selected using **validation accuracy**; the server calculates official **test accuracy** later.

If you rerun this cell, it continues training the current model. Rerun the model creation cell for a fresh run.

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
best_accuracy = -1.0
best_state = None
history = []

for epoch in range(EPOCHS):
    model.train()
    train_correct = train_total = 0
    loss_total = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        loss_total += loss.item() * len(labels)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total += len(labels)

    model.eval()
    validation_correct = validation_total = 0
    with torch.inference_mode():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            validation_correct += (logits.argmax(1) == labels).sum().item()
            validation_total += len(labels)
    val_accuracy = validation_correct / validation_total
    history.append({"epoch": epoch + 1, "loss": loss_total / train_total,
                    "train_accuracy": train_correct / train_total,
                    "validation_accuracy": val_accuracy})
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    print(f"Epoch {epoch + 1}/{EPOCHS}: loss {loss_total / train_total:.4f}, "
          f"train {train_correct / train_total:.1%}, validation {val_accuracy:.1%}")

assert best_state is not None, "Train for at least one epoch."
model.load_state_dict(best_state)
model.eval()
print(f"Best validation accuracy: {best_accuracy:.1%}")

## Record your experiments

| Architecture change | Epochs | Best validation accuracy | What did you learn? |
|---|---:|---:|---|
| Starter CNN | | | |
| Your first change | | | |
| Your next change | | | |

Which architecture would you submit, and what evidence supports your choice?

## Export and check CPU inference

This creates `submission.py` to load your architecture and saved weights, process planar RGB pixels in batches, and return class probabilities. The server runs the same code in an isolated CPU container. Keep the preprocessing consistent with your validation transform.

In [ ]:
%%writefile submission.py
"""Export alongside model.py and weights.pt; no training or downloads during inference."""

from pathlib import Path

import numpy as np
import torch
from model import CNN

_ROOT = Path(__file__).resolve().parent
_model = CNN().cpu()
_model.load_state_dict(torch.load(_ROOT / "weights.pt", map_location="cpu", weights_only=True))
_model.eval()


@torch.inference_mode()
def predict(X):
    out = []
    for start in range(0, len(X), 128):
        batch = np.array(X[start : start + 128], dtype=np.float32, copy=True)
        images = torch.from_numpy(batch.reshape(-1, 3, 32, 32)) / 255.0
        images = (images - 0.5) / 0.5
        out.append(torch.softmax(_model(images), dim=1).numpy())
    return np.concatenate(out, axis=0)


In [ ]:
assert best_state is not None, "Run training before export."
assert Path("model.py").read_text() == trained_architecture, (
    "Architecture changed since model creation. Rerun model creation and training."
)
# Save the validation-selected checkpoint as CPU tensors.
torch.save(best_state, "weights.pt")

# Check the exported code against the trained model on real validation images.
import submission
importlib.reload(submission)
sample_ids = indices[:32]
raw_rgb = validation_data.data[sample_ids]  # (N, H, W, C), uint8
raw_planar = raw_rgb.transpose(0, 3, 1, 2).reshape(-1, 3072)
probabilities = submission.predict(raw_planar)
with torch.inference_mode():
    images = torch.stack([validation_data[i][0] for i in sample_ids]).to(device)
    expected = model(images).softmax(1).cpu().numpy()
assert probabilities.shape == (32, 10)
assert np.isfinite(probabilities).all()
np.testing.assert_allclose(probabilities.sum(1), 1.0, atol=1e-5)
np.testing.assert_allclose(probabilities, expected, atol=1e-5, rtol=1e-4)

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in ("submission.py", "model.py", "weights.pt"):
        bundle.write(filename)
print("CPU inference check passed. Upload submission.zip to your arena student page.")
try:
    from google.colab import files
    files.download("submission.zip")
except ImportError:
    print("Outside Colab: download submission.zip from the notebook file browser.")

## Submit

Return to the arena student page and upload `submission.zip`. Your **test accuracy (%)** and rank appear after server evaluation. The leaderboard keeps your best successful score. Keep this notebook with your architecture, experiment table and explanations.

For evaluation failures, read the student page's error message. The server uses PyTorch on CPU, no network or torchvision, and configurable time/memory limits.